# 02 - Entendimento dos Dados da Camada Bronze

Este notebook analisa tecnicamente os arquivos Parquet da camada Bronze, gera um dicionário de dados inicial e produz insumos para a modelagem das camadas Silver e Gold.

Etapas contempladas:

1. Localizar os arquivos Parquet da Bronze.
2. Carregar as tabelas em memória.
3. Gerar resumo técnico das tabelas.
4. Gerar dicionário de dados inicial.
5. Avaliar nulos, duplicidades e cardinalidade.
6. Sugerir possíveis chaves primárias simples e compostas.
7. Sugerir possíveis relacionamentos.
8. Gerar backlog inicial para tratamento na Silver.
9. Exportar documentação em Markdown para a pasta `docs/`.

In [1]:
from pathlib import Path
from itertools import combinations

import pandas as pd

## 1. Configuração dos caminhos

In [2]:
BRONZE_PATH = Path("../data/bronze")
DOCS_PATH = Path("../docs")

DOCS_PATH.mkdir(parents=True, exist_ok=True)

print("Caminho Bronze:", BRONZE_PATH.resolve())
print("Caminho Docs:", DOCS_PATH.resolve())

Caminho Bronze: C:\Projetos\fiap-tech-challenge-fase2\data\bronze
Caminho Docs: C:\Projetos\fiap-tech-challenge-fase2\docs


## 2. Funções para localização e carga dos dados

In [3]:
def listar_arquivos_parquet(caminho_base: Path) -> list[Path]:
    """
    Lista todos os arquivos Parquet encontrados dentro de um diretório base.
    """
    if not caminho_base.exists():
        raise FileNotFoundError(f"Caminho não encontrado: {caminho_base}")

    return sorted(caminho_base.rglob("*.parquet"))


def extrair_nome_tabela(caminho_arquivo: Path) -> str:
    """
    Extrai o nome da tabela com base na estrutura:
    data/bronze/{nome_tabela}/execution_date=AAAA-MM-DD/arquivo.parquet
    """
    try:
        return caminho_arquivo.parts[-3]
    except IndexError as erro:
        raise ValueError(f"Não foi possível extrair o nome da tabela do caminho: {caminho_arquivo}") from erro


def carregar_tabelas_bronze(arquivos: list[Path]) -> dict[str, pd.DataFrame]:
    """
    Carrega os arquivos Parquet da Bronze em um dicionário de DataFrames.
    """
    tabelas = {}

    for arquivo in arquivos:
        nome_tabela = extrair_nome_tabela(arquivo)

        if nome_tabela in tabelas:
            print(f"[AVISO] Tabela duplicada encontrada: {nome_tabela}. O último arquivo será mantido.")

        tabelas[nome_tabela] = pd.read_parquet(arquivo)

    return tabelas

In [4]:
arquivos_bronze = listar_arquivos_parquet(BRONZE_PATH)

print("Arquivos Parquet encontrados:", len(arquivos_bronze))

for arquivo in arquivos_bronze:
    print(arquivo)

Arquivos Parquet encontrados: 6
..\data\bronze\alunos\execution_date=2026-06-28\alunos.parquet
..\data\bronze\meta_alfabetizacao_brasil\execution_date=2026-06-28\meta_alfabetizacao_brasil.parquet
..\data\bronze\meta_alfabetizacao_municipio\execution_date=2026-06-28\meta_alfabetizacao_municipio.parquet
..\data\bronze\meta_alfabetizacao_uf\execution_date=2026-06-28\meta_alfabetizacao_uf.parquet
..\data\bronze\municipio\execution_date=2026-06-28\municipio.parquet
..\data\bronze\uf\execution_date=2026-06-28\uf.parquet


In [5]:
tabelas_bronze = carregar_tabelas_bronze(arquivos_bronze)

print("Tabelas carregadas:", list(tabelas_bronze.keys()))

Tabelas carregadas: ['alunos', 'meta_alfabetizacao_brasil', 'meta_alfabetizacao_municipio', 'meta_alfabetizacao_uf', 'municipio', 'uf']


## 3. Resumo geral das tabelas

In [6]:
def gerar_resumo_tabelas(tabelas: dict[str, pd.DataFrame]) -> pd.DataFrame:
    """
    Gera um resumo geral com quantidade de linhas, colunas e duplicidades por tabela.
    """
    resumo = []

    for nome_tabela, df in tabelas.items():
        qtd_linhas = len(df)
        qtd_duplicados = df.duplicated().sum()

        resumo.append({
            "tabela": nome_tabela,
            "qtd_linhas": qtd_linhas,
            "qtd_colunas": len(df.columns),
            "qtd_duplicados": int(qtd_duplicados),
            "percentual_duplicados": round((qtd_duplicados / qtd_linhas) * 100, 2) if qtd_linhas > 0 else 0
        })

    return pd.DataFrame(resumo).sort_values("tabela").reset_index(drop=True)


df_resumo_tabelas = gerar_resumo_tabelas(tabelas_bronze)

df_resumo_tabelas

,tabela,qtd_linhas,qtd_colunas,qtd_duplicados,percentual_duplicados
0,alunos,3867999,13,0,0.0
1,meta_alfabetizacao_brasil,3,11,0,0.0
2,meta_alfabetizacao_municipio,10704,14,0,0.0
3,meta_alfabetizacao_uf,81,13,0,0.0
4,municipio,23995,16,0,0.0
5,uf,145,16,0,0.0


## 4. Dicionário de dados técnico inicial

In [7]:
def obter_exemplo_valor(serie: pd.Series):
    """
    Retorna um exemplo não nulo de uma coluna, quando existir.
    """
    valores_nao_nulos = serie.dropna()

    if valores_nao_nulos.empty:
        return None

    valor = valores_nao_nulos.iloc[0]

    try:
        return str(valor)
    except Exception:
        return None


def gerar_dicionario_dados(tabelas: dict[str, pd.DataFrame]) -> pd.DataFrame:
    """
    Gera um dicionário técnico de dados com informações de cada coluna.
    """
    linhas = []

    for nome_tabela, df in tabelas.items():
        qtd_linhas = len(df)

        for coluna in df.columns:
            qtd_nulos = int(df[coluna].isna().sum())
            qtd_distintos = int(df[coluna].nunique(dropna=True))

            linhas.append({
                "tabela": nome_tabela,
                "coluna": coluna,
                "tipo_dado": str(df[coluna].dtype),
                "qtd_linhas": qtd_linhas,
                "qtd_nulos": qtd_nulos,
                "percentual_nulos": round((qtd_nulos / qtd_linhas) * 100, 2) if qtd_linhas > 0 else 0,
                "qtd_valores_distintos": qtd_distintos,
                "percentual_distintos": round((qtd_distintos / qtd_linhas) * 100, 2) if qtd_linhas > 0 else 0,
                "exemplo_valor": obter_exemplo_valor(df[coluna]),
                "descricao": "",
                "observacao": ""
            })

    return pd.DataFrame(linhas)


df_dicionario = gerar_dicionario_dados(tabelas_bronze)

df_dicionario

,tabela,coluna,tipo_dado,qtd_linhas,qtd_nulos,percentual_nulos,qtd_valores_distintos,percentual_distintos,exemplo_valor,descricao,observacao
0,alunos,ano,Int64,3867999,0,0.00,2,0.00,2023,,
1,alunos,id_municipio,object,3867999,0,0.00,5548,0.14,1301902,,
2,alunos,id_municipio_nome,object,3867999,0,0.00,5279,0.14,Itacoatiara,,
3,alunos,id_escola,object,3867999,0,0.00,42811,1.11,60000747,,
4,alunos,id_aluno,object,3867999,0,0.00,2352328,60.82,13012277,,
...,...,...,...,...,...,...,...,...,...,...,...
78,uf,proporcao_aluno_nivel_4,float64,145,70,48.28,69,47.59,13.47,,
79,uf,proporcao_aluno_nivel_5,float64,145,70,48.28,67,46.21,40.41,,
80,uf,proporcao_aluno_nivel_6,float64,145,70,48.28,69,47.59,28.72,,
81,uf,proporcao_aluno_nivel_7,float64,145,70,48.28,67,46.21,3.65,,


## 5. Análise de colunas categóricas

In [8]:
def detectar_colunas_categoricas(
    tabelas: dict[str, pd.DataFrame],
    limite_distintos: int = 30
) -> pd.DataFrame:
    """
    Identifica possíveis colunas categóricas com base na quantidade de valores distintos.
    """
    linhas = []

    for nome_tabela, df in tabelas.items():
        for coluna in df.columns:
            qtd_distintos = int(df[coluna].nunique(dropna=True))

            if qtd_distintos <= limite_distintos:
                valores = df[coluna].dropna().unique().tolist()
                valores = [str(valor) for valor in valores[:50]]

                linhas.append({
                    "tabela": nome_tabela,
                    "coluna": coluna,
                    "qtd_valores_distintos": qtd_distintos,
                    "valores": valores
                })

    return pd.DataFrame(linhas).sort_values(["tabela", "qtd_valores_distintos"]).reset_index(drop=True)


df_categoricas = detectar_colunas_categoricas(tabelas_bronze)

df_categoricas

,tabela,coluna,qtd_valores_distintos,valores
0,alunos,serie,1,[2° ano do Ensino Fundamental]
1,alunos,ano,2,"[2023, 2024]"
2,alunos,presenca,2,"[Ausente, Presente]"
3,alunos,preenchimento_caderno,2,"[Prova não preenchida, Prova preenchida]"
4,alunos,alfabetizado,2,"[Não, Sim]"
5,alunos,rede,3,"[Municipal, Estadual, Privada]"
6,alunos,caderno,22,"[1, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 2,..."
7,meta_alfabetizacao_brasil,rede,1,[Pública]
8,meta_alfabetizacao_brasil,meta_alfabetizacao_2030,1,[80.0]
9,meta_alfabetizacao_brasil,meta_alfabetizacao_2024,2,"[60.0, 59.9]"


## 6. Sugestão de possíveis chaves primárias simples

In [9]:
def sugerir_chaves_primarias(tabelas: dict[str, pd.DataFrame]) -> pd.DataFrame:
    """
    Sugere possíveis chaves primárias simples com base em colunas únicas e sem nulos.
    """
    sugestoes = []

    for nome_tabela, df in tabelas.items():
        qtd_linhas = len(df)

        for coluna in df.columns:
            qtd_nulos = int(df[coluna].isna().sum())
            qtd_distintos = int(df[coluna].nunique(dropna=True))

            if qtd_linhas > 0 and qtd_nulos == 0 and qtd_distintos == qtd_linhas:
                sugestoes.append({
                    "tabela": nome_tabela,
                    "coluna_candidata": coluna,
                    "tipo": str(df[coluna].dtype),
                    "qtd_linhas": qtd_linhas,
                    "qtd_distintos": qtd_distintos,
                    "motivo": "Coluna sem nulos e com valores únicos"
                })

    return pd.DataFrame(sugestoes)


df_chaves_primarias = sugerir_chaves_primarias(tabelas_bronze)

df_chaves_primarias

,tabela,coluna_candidata,tipo,qtd_linhas,qtd_distintos,motivo
0,meta_alfabetizacao_brasil,ano,Int64,3,3,Coluna sem nulos e com valores únicos
1,meta_alfabetizacao_brasil,taxa_alfabetizacao,float64,3,3,Coluna sem nulos e com valores únicos
2,meta_alfabetizacao_brasil,percentual_participacao,float64,3,3,Coluna sem nulos e com valores únicos


## 7. Sugestão de possíveis chaves compostas

In [10]:
def sugerir_chaves_compostas(
    tabelas: dict[str, pd.DataFrame],
    colunas_prioritarias: list[str] | None = None,
    tamanho_maximo: int = 3
) -> pd.DataFrame:
    """
    Sugere possíveis chaves compostas testando combinações de colunas prioritárias.
    """
    if colunas_prioritarias is None:
        colunas_prioritarias = [
            "ano",
            "sigla_uf",
            "id_municipio",
            "rede",
            "serie",
            "id_escola",
            "id_aluno"
        ]

    sugestoes = []

    for nome_tabela, df in tabelas.items():
        qtd_linhas = len(df)

        colunas_existentes = [
            coluna for coluna in colunas_prioritarias
            if coluna in df.columns
        ]

        for tamanho in range(2, tamanho_maximo + 1):
            for combinacao in combinations(colunas_existentes, tamanho):
                df_chave = df[list(combinacao)]

                qtd_distintos = df_chave.drop_duplicates().shape[0]
                qtd_nulos = int(df_chave.isna().sum().sum())

                if qtd_linhas > 0 and qtd_distintos == qtd_linhas and qtd_nulos == 0:
                    sugestoes.append({
                        "tabela": nome_tabela,
                        "colunas_candidatas": ", ".join(combinacao),
                        "qtd_linhas": qtd_linhas,
                        "qtd_distintos_combinacao": qtd_distintos,
                        "motivo": "Combinação sem nulos e com valores únicos"
                    })

    return pd.DataFrame(sugestoes)


df_chaves_compostas = sugerir_chaves_compostas(tabelas_bronze)

df_chaves_compostas

,tabela,colunas_candidatas,qtd_linhas,qtd_distintos_combinacao,motivo
0,alunos,"ano, id_aluno",3867999,3867999,Combinação sem nulos e com valores únicos
1,alunos,"ano, id_municipio, id_aluno",3867999,3867999,Combinação sem nulos e com valores únicos
2,alunos,"ano, rede, id_aluno",3867999,3867999,Combinação sem nulos e com valores únicos
3,alunos,"ano, serie, id_aluno",3867999,3867999,Combinação sem nulos e com valores únicos
4,alunos,"ano, id_escola, id_aluno",3867999,3867999,Combinação sem nulos e com valores únicos
5,meta_alfabetizacao_brasil,"ano, rede",3,3,Combinação sem nulos e com valores únicos
6,meta_alfabetizacao_municipio,"ano, id_municipio",10704,10704,Combinação sem nulos e com valores únicos
7,meta_alfabetizacao_municipio,"ano, id_municipio, rede",10704,10704,Combinação sem nulos e com valores únicos
8,meta_alfabetizacao_uf,"ano, sigla_uf",81,81,Combinação sem nulos e com valores únicos
9,meta_alfabetizacao_uf,"ano, sigla_uf, rede",81,81,Combinação sem nulos e com valores únicos


## 8. Detecção de colunas comuns entre tabelas

In [11]:
def detectar_colunas_comuns(tabelas: dict[str, pd.DataFrame]) -> pd.DataFrame:
    """
    Detecta colunas com o mesmo nome entre tabelas diferentes.
    """
    linhas = []
    nomes_tabelas = list(tabelas.keys())

    for i, tabela_origem in enumerate(nomes_tabelas):
        for tabela_destino in nomes_tabelas[i + 1:]:
            colunas_comuns = set(tabelas[tabela_origem].columns).intersection(
                set(tabelas[tabela_destino].columns)
            )

            for coluna in sorted(colunas_comuns):
                linhas.append({
                    "tabela_origem": tabela_origem,
                    "tabela_destino": tabela_destino,
                    "coluna_comum": coluna
                })

    return pd.DataFrame(linhas)


df_colunas_comuns = detectar_colunas_comuns(tabelas_bronze)

df_colunas_comuns

,tabela_origem,tabela_destino,coluna_comum
0,alunos,meta_alfabetizacao_brasil,ano
1,alunos,meta_alfabetizacao_brasil,rede
2,alunos,meta_alfabetizacao_municipio,ano
3,alunos,meta_alfabetizacao_municipio,id_municipio
4,alunos,meta_alfabetizacao_municipio,id_municipio_nome
...,...,...,...
80,municipio,uf,proporcao_aluno_nivel_7
81,municipio,uf,proporcao_aluno_nivel_8
82,municipio,uf,rede
83,municipio,uf,serie


## 9. Sugestão de possíveis relacionamentos

In [12]:
def sugerir_relacionamentos(tabelas: dict[str, pd.DataFrame]) -> pd.DataFrame:
    """
    Sugere relacionamentos com base em chaves conhecidas do domínio.
    """
    chaves_relacionamento = [
        "sigla_uf",
        "id_municipio",
        "ano",
        "rede",
        "serie"
    ]

    relacionamentos = []

    nomes_tabelas = list(tabelas.keys())

    for tabela_origem in nomes_tabelas:
        for tabela_destino in nomes_tabelas:
            if tabela_origem == tabela_destino:
                continue

            for chave in chaves_relacionamento:
                if chave in tabelas[tabela_origem].columns and chave in tabelas[tabela_destino].columns:
                    valores_origem = set(tabelas[tabela_origem][chave].dropna().unique())
                    valores_destino = set(tabelas[tabela_destino][chave].dropna().unique())

                    if len(valores_origem) == 0 or len(valores_destino) == 0:
                        cobertura = 0
                    else:
                        cobertura = len(valores_origem.intersection(valores_destino)) / len(valores_origem) * 100

                    relacionamentos.append({
                        "tabela_origem": tabela_origem,
                        "tabela_destino": tabela_destino,
                        "chave": chave,
                        "qtd_valores_origem": len(valores_origem),
                        "qtd_valores_destino": len(valores_destino),
                        "percentual_cobertura_origem_no_destino": round(cobertura, 2)
                    })

    return pd.DataFrame(relacionamentos)


df_relacionamentos = sugerir_relacionamentos(tabelas_bronze)

df_relacionamentos.sort_values(
    ["chave", "percentual_cobertura_origem_no_destino"],
    ascending=[True, False]
)

,tabela_origem,tabela_destino,chave,qtd_valores_origem,qtd_valores_destino,percentual_cobertura_origem_no_destino
0,alunos,meta_alfabetizacao_brasil,ano,2,3,100.00
3,alunos,meta_alfabetizacao_municipio,ano,2,2,100.00
5,alunos,meta_alfabetizacao_uf,ano,2,3,100.00
8,alunos,municipio,ano,2,2,100.00
11,alunos,uf,ano,2,2,100.00
...,...,...,...,...,...,...
60,municipio,uf,serie,1,1,100.00
63,uf,alunos,serie,1,1,100.00
73,uf,municipio,serie,1,1,100.00
68,uf,meta_alfabetizacao_uf,sigla_uf,25,27,100.00


## 10. Backlog inicial para tratamento na Silver

In [13]:
def gerar_backlog_silver(df_dicionario: pd.DataFrame) -> pd.DataFrame:
    """
    Gera uma lista inicial de problemas e oportunidades de tratamento para a camada Silver.
    """
    backlog = []

    for _, linha in df_dicionario.iterrows():
        tabela = linha["tabela"]
        coluna = linha["coluna"]
        tipo_dado = linha["tipo_dado"]

        if linha["percentual_nulos"] > 0:
            backlog.append({
                "tabela": tabela,
                "coluna": coluna,
                "tipo_problema": "valor_nulo",
                "descricao": f"Coluna possui {linha['percentual_nulos']}% de valores nulos.",
                "acao_sugerida": "Avaliar preenchimento, remoção ou manutenção como nulo."
            })

        if tipo_dado == "object":
            backlog.append({
                "tabela": tabela,
                "coluna": coluna,
                "tipo_problema": "tipo_texto_generico",
                "descricao": "Coluna está como object.",
                "acao_sugerida": "Padronizar para string ou converter para tipo adequado."
            })

        if any(termo in coluna for termo in ["taxa", "percentual", "proporcao"]):
            backlog.append({
                "tabela": tabela,
                "coluna": coluna,
                "tipo_problema": "validacao_intervalo",
                "descricao": "Campo percentual/proporcional deve ser validado.",
                "acao_sugerida": "Validar se os valores estão no intervalo esperado."
            })

        if coluna.startswith("id_") or coluna in ["sigla_uf"]:
            backlog.append({
                "tabela": tabela,
                "coluna": coluna,
                "tipo_problema": "validacao_chave",
                "descricao": "Campo aparenta ser chave de relacionamento.",
                "acao_sugerida": "Validar integridade referencial entre tabelas."
            })

    return pd.DataFrame(backlog)


df_backlog_silver = gerar_backlog_silver(df_dicionario)

df_backlog_silver

,tabela,coluna,tipo_problema,descricao,acao_sugerida
0,alunos,id_municipio,tipo_texto_generico,Coluna está como object.,Padronizar para string ou converter para tipo ...
1,alunos,id_municipio,validacao_chave,Campo aparenta ser chave de relacionamento.,Validar integridade referencial entre tabelas.
2,alunos,id_municipio_nome,tipo_texto_generico,Coluna está como object.,Padronizar para string ou converter para tipo ...
3,alunos,id_municipio_nome,validacao_chave,Campo aparenta ser chave de relacionamento.,Validar integridade referencial entre tabelas.
4,alunos,id_escola,tipo_texto_generico,Coluna está como object.,Padronizar para string ou converter para tipo ...
...,...,...,...,...,...
89,uf,proporcao_aluno_nivel_6,validacao_intervalo,Campo percentual/proporcional deve ser validado.,Validar se os valores estão no intervalo esper...
90,uf,proporcao_aluno_nivel_7,valor_nulo,Coluna possui 48.28% de valores nulos.,"Avaliar preenchimento, remoção ou manutenção c..."
91,uf,proporcao_aluno_nivel_7,validacao_intervalo,Campo percentual/proporcional deve ser validado.,Validar se os valores estão no intervalo esper...
92,uf,proporcao_aluno_nivel_8,valor_nulo,Coluna possui 48.28% de valores nulos.,"Avaliar preenchimento, remoção ou manutenção c..."


## 11. Exportação do dicionário de dados para Markdown

In [14]:
def exportar_dicionario_markdown(df_dicionario: pd.DataFrame, caminho_saida: Path) -> None:
    """
    Exporta o dicionário de dados para um arquivo Markdown.
    """
    with open(caminho_saida, "w", encoding="utf-8") as arquivo:
        arquivo.write("# Dicionário de Dados - Camada Bronze\n\n")
        arquivo.write(
            "Este documento apresenta o dicionário técnico inicial das tabelas da camada Bronze.\n\n"
        )

        for tabela in sorted(df_dicionario["tabela"].unique()):
            arquivo.write(f"## Tabela: {tabela}\n\n")

            df_tabela = df_dicionario[df_dicionario["tabela"] == tabela].copy()

            colunas_exportar = [
                "coluna",
                "tipo_dado",
                "qtd_nulos",
                "percentual_nulos",
                "qtd_valores_distintos",
                "exemplo_valor",
                "descricao",
                "observacao"
            ]

            arquivo.write(df_tabela[colunas_exportar].to_markdown(index=False))
            arquivo.write("\n\n")


arquivo_dicionario = DOCS_PATH / "dicionario_dados_bronze.md"

exportar_dicionario_markdown(df_dicionario, arquivo_dicionario)

print("Dicionário exportado em:", arquivo_dicionario)

Dicionário exportado em: ..\docs\dicionario_dados_bronze.md


## 12. Exportação dos insumos de modelagem para Markdown

In [15]:
def dataframe_para_markdown_ou_mensagem(df: pd.DataFrame, mensagem_vazia: str) -> str:
    """
    Retorna uma tabela Markdown quando houver registros; caso contrário, retorna uma mensagem.
    """
    if df is None or len(df) == 0:
        return mensagem_vazia

    return df.to_markdown(index=False)


def exportar_insumos_modelagem(
    resumo_tabelas: pd.DataFrame,
    chaves_primarias: pd.DataFrame,
    chaves_compostas: pd.DataFrame,
    relacionamentos: pd.DataFrame,
    backlog_silver: pd.DataFrame,
    caminho_saida: Path
) -> None:
    """
    Exporta um documento com insumos iniciais para modelagem e camada Silver.
    """
    with open(caminho_saida, "w", encoding="utf-8") as arquivo:
        arquivo.write("# Insumos de Modelagem - Camada Bronze")

        arquivo.write("## Resumo das Tabelas")
        arquivo.write(dataframe_para_markdown_ou_mensagem(
            resumo_tabelas,
            "Nenhuma tabela carregada."
        ))
        arquivo.write("")

        arquivo.write("## Possíveis Chaves Primárias Simples")
        arquivo.write(dataframe_para_markdown_ou_mensagem(
            chaves_primarias,
            "Nenhuma chave primária simples identificada automaticamente."
        ))
        arquivo.write("")

        arquivo.write("## Possíveis Chaves Compostas")
        arquivo.write(dataframe_para_markdown_ou_mensagem(
            chaves_compostas,
            "Nenhuma chave composta identificada automaticamente."
        ))
        arquivo.write("")

        arquivo.write("## Possíveis Relacionamentos")
        arquivo.write(dataframe_para_markdown_ou_mensagem(
            relacionamentos,
            "Nenhum relacionamento identificado automaticamente."
        ))
        arquivo.write("")

        arquivo.write("## Backlog Inicial para Silver")
        arquivo.write(dataframe_para_markdown_ou_mensagem(
            backlog_silver,
            "Nenhum tratamento identificado automaticamente."
        ))
        arquivo.write("")



arquivo_modelagem = DOCS_PATH / "insumos_modelagem_bronze.md"

exportar_insumos_modelagem(
    resumo_tabelas=df_resumo_tabelas,
    chaves_primarias=df_chaves_primarias,
    chaves_compostas=df_chaves_compostas,
    relacionamentos=df_relacionamentos,
    backlog_silver=df_backlog_silver,
    caminho_saida=arquivo_modelagem
)

print("Insumos de modelagem exportados em:", arquivo_modelagem)

Insumos de modelagem exportados em: ..\docs\insumos_modelagem_bronze.md


## 13. Resumo final da análise

In [16]:
print("Resumo da análise da camada Bronze")
print("=" * 60)

print(f"Arquivos Parquet analisados: {len(arquivos_bronze)}")
print(f"Tabelas carregadas: {len(tabelas_bronze)}")
print(f"Colunas documentadas: {len(df_dicionario)}")
print(f"Possíveis chaves primárias simples: {len(df_chaves_primarias)}")
print(f"Possíveis chaves compostas: {len(df_chaves_compostas)}")
print(f"Possíveis relacionamentos sugeridos: {len(df_relacionamentos)}")
print(f"Itens no backlog inicial da Silver: {len(df_backlog_silver)}")

print("Arquivos gerados:")
print("-", arquivo_dicionario)
print("-", arquivo_modelagem)

print("Status: entendimento dos dados da Bronze concluído.")

Resumo da análise da camada Bronze
Arquivos Parquet analisados: 6
Tabelas carregadas: 6
Colunas documentadas: 83
Possíveis chaves primárias simples: 3
Possíveis chaves compostas: 12
Possíveis relacionamentos sugeridos: 74
Itens no backlog inicial da Silver: 94
Arquivos gerados:
- ..\docs\dicionario_dados_bronze.md
- ..\docs\insumos_modelagem_bronze.md
Status: entendimento dos dados da Bronze concluído.


In [18]:
# ============================================================
# Visualização simples das tabelas da camada Bronze
# ============================================================

# 1. Listar as tabelas carregadas no dicionário
print("Tabelas disponíveis na Bronze:")
print(list(tabelas_bronze.keys()))

# ------------------------------------------------------------
# 2. Escolher uma tabela para análise
# Troque o nome abaixo para analisar outra tabela:
# "uf"
# "municipio"
# "meta_alfabetizacao_brasil"
# "meta_alfabetizacao_uf"
# "meta_alfabetizacao_municipio"
# ------------------------------------------------------------

nome_tabela = "uf"

# 3. Carregar a tabela escolhida em um DataFrame
df = tabelas_bronze[nome_tabela]

# ------------------------------------------------------------
# 4. Visualizar informações gerais da tabela
# ------------------------------------------------------------

print("\nTabela selecionada:")
print(nome_tabela)

print("\nQuantidade de linhas e colunas:")
print(df.shape)

print("\nLista de colunas:")
print(list(df.columns))

# ------------------------------------------------------------
# 5. Visualizar os primeiros registros
# ------------------------------------------------------------

print("\nPrimeiros registros:")
display(df.head(20))

# ------------------------------------------------------------
# 6. Visualizar os tipos de dados
# ------------------------------------------------------------

print("\nTipos de dados:")
display(df.dtypes)

# ------------------------------------------------------------
# 7. Ver informações gerais do DataFrame
# ------------------------------------------------------------

print("\nInformações gerais:")
df.info()

# ------------------------------------------------------------
# 8. Ver quantidade de valores nulos por coluna
# ------------------------------------------------------------

print("\nQuantidade de nulos por coluna:")
display(df.isna().sum())

# ------------------------------------------------------------
# 9. Ver percentual de valores nulos por coluna
# ------------------------------------------------------------

print("\nPercentual de nulos por coluna:")
display((df.isna().sum() / len(df) * 100).round(2))

# ------------------------------------------------------------
# 10. Ver estatísticas das colunas numéricas
# ------------------------------------------------------------

print("\nEstatísticas descritivas:")
display(df.describe())

# ------------------------------------------------------------
# 11. Ver quantidade de valores distintos por coluna
# ------------------------------------------------------------

print("\nQuantidade de valores distintos por coluna:")
display(df.nunique(dropna=True))

# ------------------------------------------------------------
# 12. Analisar valores de uma coluna específica
# Troque o nome da coluna conforme a tabela escolhida
# Exemplos: "ano", "rede", "serie", "sigla_uf", "id_municipio"
# ------------------------------------------------------------

coluna_analise = "serie"

if coluna_analise in df.columns:
    print(f"\nValores únicos da coluna '{coluna_analise}':")
    display(df[coluna_analise].unique())

    print(f"\nContagem de valores da coluna '{coluna_analise}':")
    display(df[coluna_analise].value_counts(dropna=False))
else:
    print(f"\nA coluna '{coluna_analise}' não existe na tabela '{nome_tabela}'.")

# ------------------------------------------------------------
# 13. Exemplo de filtro simples
# Só executa se a coluna 'ano' existir na tabela
# ------------------------------------------------------------

if "ano" in df.columns:
    print("\nExemplo de filtro: registros do ano 2023")
    display(df[df["ano"] == 2023].head(20))

# ------------------------------------------------------------
# 14. Exemplo de filtro composto
# Só executa se as colunas 'ano' e 'rede' existirem na tabela
# ------------------------------------------------------------

if "ano" in df.columns and "rede" in df.columns:
    print("\nExemplo de filtro: ano 2023 e rede Municipal")
    display(df[(df["ano"] == 2023) & (df["rede"] == "Municipal")].head(20))

Tabelas disponíveis na Bronze:
['alunos', 'meta_alfabetizacao_brasil', 'meta_alfabetizacao_municipio', 'meta_alfabetizacao_uf', 'municipio', 'uf']

Tabela selecionada:
uf

Quantidade de linhas e colunas:
(145, 16)

Lista de colunas:
['ano', 'sigla_uf', 'sigla_uf_nome', 'serie', 'rede', 'taxa_alfabetizacao', 'media_portugues', 'proporcao_aluno_nivel_0', 'proporcao_aluno_nivel_1', 'proporcao_aluno_nivel_2', 'proporcao_aluno_nivel_3', 'proporcao_aluno_nivel_4', 'proporcao_aluno_nivel_5', 'proporcao_aluno_nivel_6', 'proporcao_aluno_nivel_7', 'proporcao_aluno_nivel_8']

Primeiros registros:


,ano,sigla_uf,sigla_uf_nome,serie,rede,taxa_alfabetizacao,media_portugues,proporcao_aluno_nivel_0,proporcao_aluno_nivel_1,proporcao_aluno_nivel_2,proporcao_aluno_nivel_3,proporcao_aluno_nivel_4,proporcao_aluno_nivel_5,proporcao_aluno_nivel_6,proporcao_aluno_nivel_7,proporcao_aluno_nivel_8
0,2023,AM,Amazonas,2° ano do Ensino Fundamental,Municipal,49.20,733.6637,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2023,PB,Paraíba,2° ano do Ensino Fundamental,Estadual,55.23,744.8152,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2023,PR,Paraná,2° ano do Ensino Fundamental,Pública (Estadual e Municipal),73.12,757.2146,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2023,AP,Amapá,2° ano do Ensino Fundamental,Municipal,41.87,732.7858,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2023,PE,Pernambuco,2° ano do Ensino Fundamental,Pública (Estadual e Municipal),58.95,747.4522,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,2023,MA,Maranhão,2° ano do Ensino Fundamental,Municipal,56.39,749.8729,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,2023,MT,Mato Grosso,2° ano do Ensino Fundamental,Estadual,48.02,738.0000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,2023,CE,Ceará,2° ano do Ensino Fundamental,Municipal,84.49,795.7293,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,2023,AL,Alagoas,2° ano do Ensino Fundamental,Pública (Estadual e Municipal),43.88,729.7227,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,2023,CE,Ceará,2° ano do Ensino Fundamental,Estadual,78.75,785.1658,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN



Tipos de dados:


ano                          Int64
sigla_uf                    object
sigla_uf_nome               object
serie                       object
rede                        object
taxa_alfabetizacao         float64
media_portugues            float64
proporcao_aluno_nivel_0    float64
proporcao_aluno_nivel_1    float64
proporcao_aluno_nivel_2    float64
proporcao_aluno_nivel_3    float64
proporcao_aluno_nivel_4    float64
proporcao_aluno_nivel_5    float64
proporcao_aluno_nivel_6    float64
proporcao_aluno_nivel_7    float64
proporcao_aluno_nivel_8    float64
dtype: object


Informações gerais:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 145 entries, 0 to 144
Data columns (total 16 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   ano                      145 non-null    Int64  
 1   sigla_uf                 145 non-null    object 
 2   sigla_uf_nome            145 non-null    object 
 3   serie                    145 non-null    object 
 4   rede                     145 non-null    object 
 5   taxa_alfabetizacao       145 non-null    float64
 6   media_portugues          145 non-null    float64
 7   proporcao_aluno_nivel_0  75 non-null     float64
 8   proporcao_aluno_nivel_1  75 non-null     float64
 9   proporcao_aluno_nivel_2  75 non-null     float64
 10  proporcao_aluno_nivel_3  75 non-null     float64
 11  proporcao_aluno_nivel_4  75 non-null     float64
 12  proporcao_aluno_nivel_5  75 non-null     float64
 13  proporcao_aluno_nivel_6  75 non-null     float64
 14  propo

...

serie
2° ano do Ensino Fundamental    145
Name: count, dtype: int64


Exemplo de filtro: registros do ano 2023


,ano,sigla_uf,sigla_uf_nome,serie,rede,taxa_alfabetizacao,media_portugues,proporcao_aluno_nivel_0,proporcao_aluno_nivel_1,proporcao_aluno_nivel_2,proporcao_aluno_nivel_3,proporcao_aluno_nivel_4,proporcao_aluno_nivel_5,proporcao_aluno_nivel_6,proporcao_aluno_nivel_7,proporcao_aluno_nivel_8
0,2023,AM,Amazonas,2° ano do Ensino Fundamental,Municipal,49.20,733.6637,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2023,PB,Paraíba,2° ano do Ensino Fundamental,Estadual,55.23,744.8152,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2023,PR,Paraná,2° ano do Ensino Fundamental,Pública (Estadual e Municipal),73.12,757.2146,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2023,AP,Amapá,2° ano do Ensino Fundamental,Municipal,41.87,732.7858,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2023,PE,Pernambuco,2° ano do Ensino Fundamental,Pública (Estadual e Municipal),58.95,747.4522,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,2023,MA,Maranhão,2° ano do Ensino Fundamental,Municipal,56.39,749.8729,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,2023,MT,Mato Grosso,2° ano do Ensino Fundamental,Estadual,48.02,738.0000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,2023,CE,Ceará,2° ano do Ensino Fundamental,Municipal,84.49,795.7293,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,2023,AL,Alagoas,2° ano do Ensino Fundamental,Pública (Estadual e Municipal),43.88,729.7227,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,2023,CE,Ceará,2° ano do Ensino Fundamental,Estadual,78.75,785.1658,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN



Exemplo de filtro: ano 2023 e rede Municipal


,ano,sigla_uf,sigla_uf_nome,serie,rede,taxa_alfabetizacao,media_portugues,proporcao_aluno_nivel_0,proporcao_aluno_nivel_1,proporcao_aluno_nivel_2,proporcao_aluno_nivel_3,proporcao_aluno_nivel_4,proporcao_aluno_nivel_5,proporcao_aluno_nivel_6,proporcao_aluno_nivel_7,proporcao_aluno_nivel_8
0,2023,AM,Amazonas,2° ano do Ensino Fundamental,Municipal,49.20,733.6637,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2023,AP,Amapá,2° ano do Ensino Fundamental,Municipal,41.87,732.7858,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,2023,MA,Maranhão,2° ano do Ensino Fundamental,Municipal,56.39,749.8729,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,2023,CE,Ceará,2° ano do Ensino Fundamental,Municipal,84.49,795.7293,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
10,2023,MS,Mato Grosso do Sul,2° ano do Ensino Fundamental,Municipal,47.53,736.0729,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
13,2023,SC,Santa Catarina,2° ano do Ensino Fundamental,Municipal,62.04,747.7686,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
19,2023,MT,Mato Grosso,2° ano do Ensino Fundamental,Municipal,55.43,747.3200,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
20,2023,GO,Goiás,2° ano do Ensino Fundamental,Municipal,66.72,752.1315,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
21,2023,PA,Pará,2° ano do Ensino Fundamental,Municipal,47.71,732.6697,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
25,2023,SP,São Paulo,2° ano do Ensino Fundamental,Municipal,51.28,739.4069,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [19]:
# ============================================================
# Exportar todas as tabelas Bronze para CSV delimitado por ;
# com cuidado para acentuação e caracteres especiais no Excel
# ============================================================

from pathlib import Path
import pandas as pd

# Pasta onde os CSVs serão gerados
EXPORT_PATH = Path("../data/export/bronze_csv")
EXPORT_PATH.mkdir(parents=True, exist_ok=True)

# Configurações do CSV
# sep=";"              -> separador comum no padrão brasileiro
# encoding="utf-8-sig" -> evita problemas de acentuação ao abrir no Excel
# decimal=","          -> mantém casas decimais no padrão brasileiro
# index=False          -> não exporta o índice do DataFrame
for nome_tabela, df in tabelas_bronze.items():
    caminho_csv = EXPORT_PATH / f"{nome_tabela}.csv"

    df.to_csv(
        caminho_csv,
        sep=";",
        index=False,
        encoding="utf-8-sig",
        decimal=","
    )

    print(f"[OK] {nome_tabela} exportada para {caminho_csv}")

print("\nExportação concluída.")
print(f"Arquivos salvos em: {EXPORT_PATH.resolve()}")

[OK] alunos exportada para ..\data\export\bronze_csv\alunos.csv
[OK] meta_alfabetizacao_brasil exportada para ..\data\export\bronze_csv\meta_alfabetizacao_brasil.csv
[OK] meta_alfabetizacao_municipio exportada para ..\data\export\bronze_csv\meta_alfabetizacao_municipio.csv
[OK] meta_alfabetizacao_uf exportada para ..\data\export\bronze_csv\meta_alfabetizacao_uf.csv
[OK] municipio exportada para ..\data\export\bronze_csv\municipio.csv
[OK] uf exportada para ..\data\export\bronze_csv\uf.csv

Exportação concluída.
Arquivos salvos em: C:\Projetos\fiap-tech-challenge-fase2\data\export\bronze_csv
